# Practice 4 — Production Architecture, APIs, Observability, and Reliability

Observability, SLOs, API contracts, durable idempotency.

**How it works:** each exercise is almost-finished code with a few `___` blanks. Fill them in, run the cell, then run the `check(...)` cell below it. ✅ means move on. ❌ shows your result and the test input, never the answer, and failing twice unlocks a hint.

Prefer a terminal? The same exercises are in [`practice/exercises/advanced_04_production_systems.py`](../../practice/exercises/advanced_04_production_systems.py).

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "exercises"))
    sys.path.insert(0, os.getcwd())

## Setup

Run this cell first.

In [ ]:
from __future__ import annotations
from check import Checker, ___  # ___ marks the blanks you fill in

check = Checker('advanced_04_production_systems')

import json
import math
import sqlite3
import statistics
import uuid
from collections.abc import Callable
from datetime import datetime, timezone

## Exercises

### 1. `make_event`

Return {'version': 1, 'code', 'correlation_id', 'timestamp'} with an ISO-8601 UTC timestamp.

In [ ]:
def make_event(code: str, correlation_id: str | None = None,
               clock: Callable[[], datetime] = lambda: datetime.now(timezone.utc)) -> dict:
    """Return {'version': 1, 'code', 'correlation_id', 'timestamp'} with an ISO-8601 UTC timestamp.

    A missing correlation_id is generated with uuid4. The clock is injectable for tests.
    """
    return ___

In [ ]:
check('make_event')

### 2. `latency_summary`

Return {'count', 'median', 'p95'}; p95 uses the nearest-rank method.

In [ ]:
def latency_summary(milliseconds: list[float]) -> dict[str, float]:
    """Return {'count', 'median', 'p95'}; p95 uses the nearest-rank method.

    nearest rank: sort, then take the value at position ceil(0.95 * n) (1-based).
    Raise ValueError for an empty list.
    """
    if ___:
        raise ValueError("no samples")
    ordered = sorted(milliseconds)
    rank = math.ceil(0.95 * len(ordered))
    return ___

In [ ]:
check('latency_summary')

### 3. `availability`

Success ratio as a percentage rounded to 3 decimals; 100.0 when total is 0.

In [ ]:
def availability(successes: int, total: int) -> float:
    """Success ratio as a percentage rounded to 3 decimals; 100.0 when total is 0."""
    if ___:
        return 100.0
    return round(100 * successes / total, 3)

In [ ]:
check('availability')

### 4. `error_budget`

How many requests may fail while still meeting the target (round down).

In [ ]:
def error_budget(target_percent: float, requests: int) -> int:
    """How many requests may fail while still meeting the target (round down).

    error_budget(99.9, 1_000_000) == 1000
    """
    return ___

In [ ]:
check('error_budget')

### 5. `serialize_user`

v1 -> {'id', 'name'}; v2 adds optional 'email' (None if absent). Other versions: ValueError.

In [ ]:
def serialize_user(user: dict, version: int = 1) -> dict:
    """v1 -> {'id', 'name'}; v2 adds optional 'email' (None if absent). Other versions: ValueError.

    v1 output must never change: old clients depend on it.
    """
    if ___:
        raise ValueError(f"unsupported version {version}")
    body = {"id": user["id"], "name": user["name"]}
    if ___:
        body["email"] = user.get("email")
    return body

In [ ]:
check('serialize_user')

### 6. `error_envelope`

Build {'error': {'code', 'message', 'status'}}. For status >= 500 hide internal details:

In [ ]:
def error_envelope(code: str, message: str, status: int) -> dict:
    """Build {'error': {'code', 'message', 'status'}}. For status >= 500 hide internal details:
    message becomes 'Internal server error'."""
    safe_message = "Internal server error" if status >= 500 else message
    return ___

In [ ]:
check('error_envelope')

### 7. `iterate_cursor`

Follow {'items': [...], 'cursor': next or None} starting from fetch(None).

In [ ]:
def iterate_cursor(fetch: Callable[[str | None], dict]) -> list:
    """Follow {'items': [...], 'cursor': next or None} starting from fetch(None).

    Raise RuntimeError if a cursor repeats (the server is looping).
    """
    items, seen, cursor = [], set(), None
    while ___:
        page = fetch(cursor)
        items.extend(page["items"])
        cursor = page.get("cursor")
        if ___:
            return items
        if ___:
            raise RuntimeError(f"cursor repeated: {cursor}")
        seen.add(cursor)

In [ ]:
check('iterate_cursor')

### 8. `IdempotencyStore`

SQLite-backed idempotency: execute(key, action) runs action() at most once per key

In [ ]:
class IdempotencyStore:
    """SQLite-backed idempotency: execute(key, action) runs action() at most once per key
    and stores its JSON-serializable result; later calls return the stored result.

    If action() raises, nothing is stored, so a retry can run it again.
    """

    def __init__(self, conn: sqlite3.Connection):
        self.conn = conn
        with ___:
            conn.execute("CREATE TABLE IF NOT EXISTS idempotency (key TEXT PRIMARY KEY, result TEXT NOT NULL)")

    def execute(self, key: str, action: Callable[[], object]):
        row = self.conn.execute("SELECT result FROM idempotency WHERE key = ?", (key,)).fetchone()
        if ___:
            return ___
        result = action()
        with self.conn:
            self.conn.execute("INSERT INTO idempotency (key, result) VALUES (?, ?)", (key, json.dumps(result)))
        return result

In [ ]:
check('IdempotencyStore')

### 9. `should_alert`

Alert only if the last `sustained` samples are ALL above threshold (avoids flapping).

In [ ]:
def should_alert(error_rates: list[float], threshold: float, sustained: int) -> bool:
    """Alert only if the last `sustained` samples are ALL above threshold (avoids flapping)."""
    recent = error_rates[-sustained:]
    return ___

In [ ]:
check('should_alert')

### 10. `readiness`

Run every check; a check that raises counts as failed. Return

In [ ]:
def readiness(checks: dict[str, Callable[[], bool]]) -> dict:
    """Run every check; a check that raises counts as failed. Return
    {'ready': all passed, 'checks': {name: 'ok' | 'fail'}}."""
    results = {}
    for name, probe in ___:
        try:
            results[name] = "ok" if probe() else "fail"
        except Exception:
            results[name] = "fail"
    return ___

In [ ]:
check('readiness')

### 11. `migrate`

Apply migrations[i] for every i >= the current PRAGMA user_version, each in its own

In [ ]:
def migrate(conn: sqlite3.Connection, migrations: list[str]) -> int:
    """Apply migrations[i] for every i >= the current PRAGMA user_version, each in its own
    transaction, bumping user_version after each. Return the final version. Re-running is a no-op."""
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    for index in ___:
        with conn:
            conn.execute(migrations[index])
            conn.execute(f"PRAGMA user_version = {index + 1}")
    return ___

In [ ]:
check('migrate')

### 12. `parse_version`

Parse 'MAJOR.MINOR.PATCH' (optional leading 'v') into ints; ValueError otherwise.

In [ ]:
def parse_version(text: str) -> tuple[int, int, int]:
    """Parse 'MAJOR.MINOR.PATCH' (optional leading 'v') into ints; ValueError otherwise."""
    parts = text.removeprefix("v").split(".")
    if ___:
        raise ValueError(f"not a semantic version: {text!r}")
    return ___

In [ ]:
check('parse_version')

### 13. `is_compatible`

A client works with a server of the same MAJOR version and an equal or newer MINOR.

In [ ]:
def is_compatible(client: str, server: str) -> bool:
    """A client works with a server of the same MAJOR version and an equal or newer MINOR."""
    c, s = parse_version(client), parse_version(server)
    return ___

In [ ]:
check('is_compatible')

## Finished?

In [ ]:
check.summary()